# Bloque III — Clasificación, Matriz de Confusión y ROC
## Ejercicio integrador — Solución comentada

**Dataset:** `clientes_abandono_mayo_2026.csv`  
**Objetivo:** Predecir el abandono de clientes y elegir el umbral de decisión óptimo.


In [ ]:
# Configuración común
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score,
    classification_report
)

## 1–8. Preparación de datos y entrenamiento de modelos

*(Igual que el notebook principal — se incluye comprimido para centrar el foco en el ejercicio)*

In [ ]:
df = pd.read_csv("../data/clientes_abandono_mayo_2026.csv")

target = "abandono"
features_num = ["edad", "ingresos", "compras_12m", "visitas_web",
                "reclamaciones", "antiguedad_meses", "ticket_medio"]
features_cat = ["segmento"]

X = df[features_num + features_cat]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, features_num),
    ("cat", categorical_transformer, features_cat)
])

def evaluar_clasificador(nombre, modelo, X_train, X_test, y_train, y_test):
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    if hasattr(modelo, "predict_proba"):
        proba = modelo.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, proba)
    else:
        proba = None
        auc = np.nan
    return {
        "modelo":     nombre,
        "accuracy":   accuracy_score(y_test, pred),
        "precision":  precision_score(y_test, pred, zero_division=0),
        "recall":     recall_score(y_test, pred, zero_division=0),
        "f1":         f1_score(y_test, pred, zero_division=0),
        "roc_auc":    auc
    }, pred, proba

logit = Pipeline([("preprocessor", preprocessor),
                  ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])
tree  = Pipeline([("preprocessor", preprocessor),
                  ("model", DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced"))])
rf    = Pipeline([("preprocessor", preprocessor),
                  ("model", RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"))])

res_logit, pred_logit, proba_logit = evaluar_clasificador("Logistic Regression", logit, X_train, X_test, y_train, y_test)
res_tree,  pred_tree,  proba_tree  = evaluar_clasificador("Decision Tree",        tree,  X_train, X_test, y_train, y_test)
res_rf,    pred_rf,    proba_rf    = evaluar_clasificador("Random Forest",        rf,    X_train, X_test, y_train, y_test)

---
## Ejercicio — Solución

### Pregunta 1: Comparar los tres modelos por recall

El **recall** (sensibilidad) mide qué proporción de los abandonos reales conseguimos detectar.  
Es la métrica prioritaria cuando el coste de no detectar un evento es alto.


In [ ]:
# Ordenamos por recall de mayor a menor
tabla_recall = pd.DataFrame([res_logit, res_tree, res_rf]).sort_values("recall", ascending=False)
display(tabla_recall)

# Visualización
fig, ax = plt.subplots(figsize=(7, 4))
modelos_nombres = tabla_recall["modelo"].tolist()
valores_recall  = tabla_recall["recall"].tolist()
colores = ["#378ADD", "#1D9E75", "#D85A30"]

bars = ax.barh(modelos_nombres, valores_recall, color=colores, edgecolor="white")
ax.set_xlabel("Recall")
ax.set_title("Comparación de recall por modelo")
ax.set_xlim(0, 1.1)
for bar, val in zip(bars, valores_recall):
    ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

**Interpretación:**

- La **Regresión Logística** obtiene el mayor recall (~0.59): detecta más de la mitad de los abandonos.
- El **Árbol de Decisión** le sigue de cerca (~0.55).
- El **Random Forest** con umbral 0.5 tiene recall muy bajo (~0.24): es muy conservador y "prefiere" no alertar si no está seguro.

En problemas con clases desbalanceadas (solo ~33 % de abandonos), el Random Forest tiende a sesgar hacia la clase mayoritaria salvo que se baje el umbral.


### Preguntas 2 y 3: Ajustar el umbral del Random Forest

Por defecto el umbral es 0.5. Bajarlo aumenta el recall (detectamos más abandonos)
pero reduce la precisión (más falsas alarmas). Es un trade-off que depende del negocio.


In [ ]:
resultados_umbral = []
for umbral in [0.2, 0.3, 0.4, 0.5, 0.6]:
    pred_u = (proba_rf >= umbral).astype(int)
    resultados_umbral.append({
        "umbral":    umbral,
        "precision": round(precision_score(y_test, pred_u, zero_division=0), 3),
        "recall":    round(recall_score(y_test, pred_u, zero_division=0), 3),
        "f1":        round(f1_score(y_test, pred_u, zero_division=0), 3)
    })

tabla_umbrales = pd.DataFrame(resultados_umbral)
display(tabla_umbrales)

# Gráfico trade-off
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tabla_umbrales["umbral"], tabla_umbrales["precision"],
        "o-", color="#378ADD", label="Precision", linewidth=2)
ax.plot(tabla_umbrales["umbral"], tabla_umbrales["recall"],
        "s-", color="#D85A30", label="Recall", linewidth=2)
ax.plot(tabla_umbrales["umbral"], tabla_umbrales["f1"],
        "^--", color="#1D9E75", label="F1", linewidth=1.5, alpha=0.9)
ax.axvline(0.3, color="gray", linestyle=":", alpha=0.7, label="Umbral elegido (0.3)")
ax.set_xlabel("Umbral de decisión")
ax.set_title("Trade-off Precision / Recall / F1 — Random Forest")
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

**Umbral elegido: 0.3**

Con umbral = 0.3:
- **Recall ≈ 0.64** — detectamos el 64 % de los clientes que van a abandonar.
- **Precision ≈ 0.38** — el 38 % de las alertas son correctas (62 % son falsas alarmas).
- Comparado con umbral 0.5: el recall sube de 0.25 a 0.64 a un coste asumible de precisión.

El umbral 0.2 ofrece recall aún mayor (~0.90) pero la precisión baja a 0.36 y se generarían
demasiadas intervenciones innecesarias. El 0.3 es un equilibrio razonable.


### Pregunta 4: Matriz de confusión para umbral 0.3


In [ ]:
UMBRAL_ELEGIDO = 0.3
pred_final = (proba_rf >= UMBRAL_ELEGIDO).astype(int)

cm = confusion_matrix(y_test, pred_final)
tn, fp, fn, tp = cm.ravel()

print(f"Verdaderos Negativos (TN): {tn}  — no abandonan, correctamente clasificados")
print(f"Falsos Positivos     (FP): {fp}  — no abandonan, pero el modelo alerta")
print(f"Falsos Negativos     (FN): {fn}  — SÍ abandonan, pero el modelo NO alerta  ← más costoso")
print(f"Verdaderos Positivos (TP): {tp}  — abandonan, correctamente detectados")
print(f"\nRecall  = TP / (TP + FN) = {tp}/{tp+fn} = {tp/(tp+fn):.3f}")
print(f"Precision = TP / (TP + FP) = {tp}/{tp+fp} = {tp/(tp+fp):.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["No abandono", "Abandono"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Matriz de confusión — Random Forest (umbral={UMBRAL_ELEGIDO})")
plt.tight_layout()
plt.show()

### Pregunta 5: Conclusión ejecutiva

**Contexto:** Se ha entrenado un modelo de clasificación para predecir qué clientes
van a abandonar en los próximos meses, usando variables de comportamiento de compra,
reclamaciones y antigüedad.

**Resultados principales:**

| Métrica | Regresión Logística | Árbol de Decisión | Random Forest (umbral=0.3) |
|---|---|---|---|
| Recall | ~0.59 | ~0.55 | **0.64** |
| Precision | ~0.44 | ~0.42 | 0.38 |
| AUC-ROC | ~0.66 | ~0.61 | ~0.62 |

El **Random Forest con umbral 0.3** ofrece el mejor recall (64 % de abandonos detectados),
que es la métrica prioritaria en este problema. Su AUC-ROC no es el más alto, pero ajustando
el umbral se consigue un comportamiento operativo superior.

**Recomendación de negocio:**

Activar el modelo con umbral 0.3. De cada 400 clientes del test, el sistema detecta
correctamente 85 abandonos y genera 137 falsas alarmas. Para una intervención de coste
bajo (correo, descuento de retención), eso es económicamente rentable siempre que el
valor de retener a un cliente supere el coste de contactar a ~1.6 clientes en falsa alarma.

**¿Qué error es más costoso: falso positivo o falso negativo?**

El **falso negativo** es más costoso. No detectar a un cliente que va a irse implica
perder toda su facturación futura (Customer Lifetime Value). El falso positivo, en cambio,
solo supone el coste marginal de una acción de retención. Por eso tiene sentido aceptar
más falsas alarmas a cambio de un mayor recall.


In [ ]:
# Curva ROC comparativa para los tres modelos
fig, ax = plt.subplots(figsize=(6, 5))

for proba, nombre, color in [
    (proba_logit, "Logistic Regression", "#378ADD"),
    (proba_tree,  "Decision Tree",       "#1D9E75"),
    (proba_rf,    "Random Forest",       "#D85A30"),
]:
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{nombre} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5, label="Clasificador aleatorio")
ax.set_xlabel("Tasa de Falsos Positivos (FPR)")
ax.set_ylabel("Tasa de Verdaderos Positivos (TPR / Recall)")
ax.set_title("Curvas ROC — Comparativa de modelos")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
